# WinCLIP Colab driver

Companion to `docs/superpowers/plans/2026-07-24-winclip-zeroshot-adapter.md` (read its two
amendments first). Cells are numbered so a session can be driven by number.

**Phase 0** — environment. **Phase 1** — API verification: no dataset, no download, and it does
not need a GPU. **Phase 2 onward** needs `src/vlmab/methods/winclip_backend.py`, which does not
exist yet; the runners refuse `--method winclip` until it does, by design.

**The cell order is load-bearing.** anomalib runs `try: import open_clip` at **module import
time** and caches `None` for the whole session if it fails. Importing anomalib before
`anomalib[clip]` is installed therefore poisons the kernel, and only a restart clears it — that
cost the 2026-10-02 session a restart mid-phase. So: install everything (0.1, 0.2), **restart**,
then import (0.3).


## Phase 0 — environment


In [ ]:
# 0.1 — GPU check, clone, editable install. NOTHING here imports anomalib, on purpose.
# A GPU is optional for Phase 1; it is required from Phase 2 on.
!nvidia-smi -L || echo "no GPU in this runtime — fine for Phase 1, which needs none"

REPO = "/content/vlm-anomaly-bench"
import os, sys
if not os.path.isdir(REPO):
    !git clone https://github.com/andrudebaran7/vlm-anomaly-bench.git {REPO}
%cd {REPO}

# NOT --quiet: a failed editable install surfaces three cells later as a baffling
# "No module named 'vlmab'", and the quiet flag is what hides the real reason.
!pip install -e .

if REPO + "/src" not in sys.path:
    sys.path.insert(0, REPO + "/src")
print("\nrepo ready at", REPO)

In [ ]:
# 0.2 — anomalib WITH the clip extra. This is the step the 2026-10-02 session was missing:
# plain `anomalib==2.6.0` installs fine and then WinClip raises
#   ImportError: open_clip is required for VLM models
# `anomalib[clip]` is anomalib's OWN declared extra, so it pulls the open-clip version anomalib
# was tested against rather than one we guessed. Note it is a RANGE
# (open-clip-torch<2.26.1,>=2.23.0), not a pin — 0.3 records what it actually resolved to.
!pip install "anomalib[clip]==2.6.0"

print("\n" + "=" * 72)
print("RESTART THE RUNTIME NOW:   Runtime -> Restart session")
print("Then run 0.3, and nothing before it.")
print()
print("Why: anomalib's open_clip guard is module-level. If anomalib was imported in this")
print("kernel before open_clip resolved, it cached a failure that no later install can undo.")
print("=" * 72)

### ⚠️ Restart here

`Runtime` → `Restart session`. The `%cd` is lost; 0.3 restores it. 0.3 is idempotent — safe to
re-run any number of times, at any point.


In [ ]:
# 0.3 — post-restart recovery and the version record. Idempotent.
REPO = "/content/vlm-anomaly-bench"
%cd {REPO}
import sys
if REPO + "/src" not in sys.path:
    sys.path.insert(0, REPO + "/src")

from importlib.metadata import version
import torch
import anomalib
import open_clip          # must import silently. If THIS fails, 0.2 did not take — rerun it.
import vlmab

print("anomalib         ", anomalib.__version__)
print("open-clip-torch  ", version("open-clip-torch"), "  (anomalib asks for <2.26.1,>=2.23.0)")
print("torch            ", torch.__version__)
print("cuda available   ", torch.cuda.is_available())
print("vlmab from       ", vlmab.__file__)

assert anomalib.__version__ == "2.6.0", "the anomalib pin did not take; rerun 0.2 and restart"
print("\nRECORD the first three — they are provenance for configs/methods/winclip.yaml.")
print("NOTE: PatchCore's eight categories were produced on torch 2.11.0+cu128. If the line")
print("above differs, see docs/next-steps.md, \"Colab's torch build has moved\".")

## Phase 1 — API verification

No dataset, no download, no GPU required. Each cell answers a question that changes what the
Phase 2 backend has to do, so none of them is a formality.

**1.2 and 1.4 are DISCOVERY, not verified API.** Nobody has read anomalib 2.6.0's `WinClip`
internals; the attribute names are guesses and the loops are there to replace them with what the
object really exposes. Protocol §3 forbids writing an upstream call as if it were verified, and
2.6.0 has already diverged from its own documentation in five recorded ways. An `AttributeError`
or a `None` in those cells is the expected first outcome — read the object, do not assume the cell
is broken.


In [ ]:
# 1.1 — VERIFY the WinClip constructor and its pre-processor.
import inspect
from anomalib.models import WinClip

print("WinClip.__init__:\n ", inspect.signature(WinClip.__init__), "\n")
print("pre-processor transform:\n ", WinClip.configure_pre_processor().transform)

# Expect: k_shot=0 as the zero-shot default; class_name and scales accepted; scales=(2,3).
# The TRANSFORM is the load-bearing part. AnomalibModule.forward runs self.pre_processor
# UNCONDITIONALLY, so whatever prints here is what winclip_backend.score() must NOT apply a
# second time. PatchCore double-normalised for weeks because the operation is monotonic, so it
# preserved ranking and every test passed.

In [ ]:
# 1.2 — THE COUNT, from the installed module rather than a dir() hunt.
# Read from anomalib 2.6.0's source on 2026-10-03 (anomalib/models/image/winclip/prompting.py).
# The earlier dir() loop crashed on WinClipModel.patch_embeddings, a property that RAISES
# RuntimeError when empty — getattr's default only swallows AttributeError.
# Paper (Fig. 6, arXiv:2303.14814 p.12): 7 normal + 4 anomaly state words x 22 templates
# = 154 normal / 88 anomaly. Expected here: 147 / 84, with one template listed twice.
from anomalib.models.image.winclip import prompting as P

normal, anomalous = P.create_prompt_ensemble("vial")
print("state words   normal", len(P.NORMAL_STATES), "| anomaly", len(P.ANOMALOUS_STATES), "   (paper: 7 | 4)")
print("templates    ", len(P.TEMPLATES), "| distinct", len(set(P.TEMPLATES)), "   (paper: 22 | 22)")
print("prompts       normal", len(normal), f"(distinct {len(set(normal))})",
      "| anomaly", len(anomalous), f"(distinct {len(set(anomalous))})", "   (paper: 154 | 88)")
print("listed twice:", sorted({t for t in P.TEMPLATES if P.TEMPLATES.count(t) > 1}))

PAPER_MISSING = ["a cropped photo of a {}.", "a jpeg corrupted photo of a {}."]
print("paper templates absent here:", [t for t in PAPER_MISSING if t not in P.TEMPLATES])


In [ ]:
# 1.3 — The text embeddings exist only after setup, and are ONE MEAN PER CLASS.
# WinClip(class_name=...) stores the name but builds WinClipModel() WITHOUT it, so
# m.model.class_name is None until WinClip.setup(stage) or m.model.setup(name) runs.
# Phase B's backend must do one of the two per category (the name is the prompt noun).
from anomalib.models import WinClip

m = WinClip(class_name="vial")
inner = m.model
print("after construction: inner.class_name =", inner.class_name)
inner.setup("vial")
print("after inner.setup('vial'): class_name =", inner.class_name,
      "| text_embeddings", tuple(inner.text_embeddings.shape), " (expected (2, D))")


In [ ]:
# 1.4 — WHICH CLIP WEIGHTS (priority-2 risk #2).
# The paper, read from the PDF 2026-09-18: "we use the LAION-400M based CLIP with ViT-B/16+"
# (arXiv:2303.14814 §5). configs/methods/winclip.yaml carries
#   pretrained_resolved: unverified_until_first_colab_run
# Same architecture with different pre-training is a DIFFERENT MODEL, and WinCLIP is entirely a
# function of its frozen features.
import open_clip
import anomalib.models.image.winclip.torch_model as tm

print("anomalib's module constants:")
for k, v in sorted(vars(tm).items()):
    if k.isupper() and isinstance(v, (str, int, float, tuple, list)):
        print(f"  {k:18} = {v!r}")

print("\nopen_clip tags available for that architecture:")
try:
    print(" ", open_clip.list_pretrained_tags_by_model("ViT-B-16-plus-240"))
except Exception as exc:
    print("  (could not list:", exc, ")")

print("\nfrom the constructed model:")
for attr in ("backbone", "pretrained", "model_name"):
    print(f"  inner.{attr} =", getattr(inner, attr, None))

### After 1.4 — provenance written down ✅ (2026-10-03)

`configs/methods/winclip.yaml` now records anomalib 2.6.0, open-clip-torch 2.24.0, torch
2.11.0+cu130, the 240 BICUBIC + CLIP-Normalize transform, `pretrained_resolved: laion400m_e31`,
and both prompt counts (anomalib 147/84, paper 154/88). The template decision is protocol §3
**v0.2.19**: the backend installs the paper's 22 templates before any text embedding is built.
All of it was committed before Phase 2 scores anything.


## Phase 2 — the backend, before any dataset

`src/vlmab/methods/winclip_backend.py` exists (2026-10-03). Three cells, no dataset, a few minutes:

* **2.0** pulls the repo, so this VM has the backend. Run it even on a VM that ran Phase 1.
* **2.1** builds the backend and scores two synthetic images. It checks **shape, finiteness and
  the template patch — not separation**: zero-shot WinCLIP has no reason to separate a white
  square on noise, and asserting it would be a test of nothing.
* **2.2** measures which transform `forward()` really applies. Reading the source says it is
  `export_transform` — the same resize **without antialiasing** — not the `transform` printed in
  1.1. Probe stage 11 could not see this (256x256 images). Hypothesis and its consequences for
  PatchCore are in `docs/next-steps.md`, written before this cell ran.


In [ ]:
# 2.0 — pull the backend onto this VM. The install is editable, so a pull is enough,
# PROVIDED vlmab.methods.winclip_backend has not been imported in this kernel yet.
!git -C /content/vlm-anomaly-bench pull --ff-only
!git -C /content/vlm-anomaly-bench log --oneline -1


In [ ]:
# 2.1 — backend smoke test. Shape, finiteness, templates. NOT separation (see above).
import numpy as np
from anomalib.models.image.winclip import prompting
from vlmab.methods.winclip_backend import WinClipBackend

b = WinClipBackend(seed=0)
print("prompt counts   ", b.prompt_counts, " (must be 154 / 88)")
print("TEMPLATES       ", len(prompting.TEMPLATES), "| distinct", len(set(prompting.TEMPLATES)), " (must be 22 | 22)")
print("post_processor  ", b._model.post_processor, " (must be None: scores raw by construction)")

rng = np.random.default_rng(0)
normal = rng.integers(90, 110, (480, 640, 3), dtype=np.uint8)   # non-square on purpose
anom = normal.copy(); anom[100:200, 200:300] = 255

s_n, m_n = b.score(normal, "object")
s_a, m_a = b.score(anom, "object")
s_v, _ = b.score(anom, "vial")          # another noun -> other text embeddings -> another score
print(f"object: normal {s_n:.6f} | anomalous {s_a:.6f} | map {m_a.shape} {m_a.dtype}")
print(f"vial  : anomalous {s_v:.6f}  (must differ from the 'object' score: the noun reached the prompts)")
assert b.prompt_counts == {"normal": 154, "anomaly": 88}
assert m_a.ndim == 2 and np.isfinite(m_a).all() and np.isfinite([s_n, s_a, s_v]).all()
assert s_v != s_a, "changing the category did not change the score: text embeddings not rebuilt"
print("\n2.1 OK — paste the four lines above.")


In [ ]:
# 2.2 — WHICH transform does forward() apply? MEASURED, three ways, on a 480x640 image.
# Read from the anomalib 2.6.0 source (2026-10-03): PreProcessor.forward applies
# `export_transform`, a copy of `transform` with antialiasing DISABLED on every Resize, built once
# at construction. The Lightning loops apply `transform`. Probe stage 11 could not see the
# difference: its images were 256x256, so its Resize was the identity.
# HYPOTHESIS recorded in docs/next-steps.md BEFORE this cell ran. Prediction:
#   forward == export path, and the transform path DIFFERS.
import torch

m = b._model
pre = m.pre_processor
print("transform        :", pre.transform)
print("export_transform :", pre.export_transform)

x = torch.from_numpy(anom).permute(2, 0, 1).float().div(255.0).to("cuda")   # 3x480x640, raw
def s(module, inp):
    with torch.no_grad():
        return float(module(inp.unsqueeze(0)).pred_score.reshape(-1)[0].item())

via_forward   = s(m, x)                                     # what score() does
via_export    = s(m.model, pre.export_transform(x))         # antialias OFF
via_transform = s(m.model, pre.transform(x))                # antialias ON (Lightning loops)
twice         = s(m.model, pre.transform(pre.transform(x))) # the double-normalisation bug
print(f"\nforward(raw)                 {via_forward:.8f}")
print(f"model(export_transform(raw)) {via_export:.8f}   predicted: EQUAL to forward")
print(f"model(transform(raw))        {via_transform:.8f}   predicted: DIFFERENT")
print(f"model(transform x2 (raw))    {twice:.8f}   the old bug, for scale")
print(f"\n|forward - transform path| = {abs(via_forward - via_transform):.3e}")
print("Paste all of it. Nothing is asserted: this cell is a measurement, and the decision it")
print("feeds is recorded after the numbers, not before.")


## Phase 3 onward — scripts, not cells

Per the descope spec (protocol v0.2.18), **a measured cost probe comes first**: time one category
before planning the gate sessions. Then:

| what | command |
|---|---|
| determinism measurement (A.4) | one category twice at two seeds; bit-identical scores mean one run is reportable |
| MVTec AD classic gate — 91.8 ± 1.0 | `!python scripts/run_mvtec_ad.py --method winclip` |
| VisA gate — 78.1 ± 1.0 | `!python scripts/run_visa_secondary.py --method winclip` |
| MVTec AD 2 grid | `!python scripts/run_mvtec_ad2.py --category <c> --method winclip` |

Both of WinCLIP's figures **gate** (protocol §2 v0.2.14), unlike PatchCore's VisA secondary.
Neither gate dataset needs a manual upload. Do not pass `--which` by hand — the block is derived
from the config by `vlmab.eval.targets.block_for`, which is what keeps `not_the_target` (Table 7's
78.9, higher than the real 78.1) from ever being scored against.
